# 09_vqe_noise_and_optimization_sensitivity

**Purpose:**

- Demonstrate how finite-shot (sampling) noise makes VQE (Variational Quantum Eigensolver) behavior stochastic
- Compare optimizer sensitivity under noise (COBYLA vs SPSA) with identical Hamiltonian / ansatz / shots
- Contrast finite-shot results with an exact (statevector) baseline

**Core idea:**

VQE minimizes the energy expectation value ⟨ψ(θ)|H|ψ(θ)⟩ over a restricted, parameterized family of states defined by an ansatz.
With finite shots, ⟨H⟩ is estimated from sampled measurement outcomes, making the objective noisy and changing optimizer behavior.


In [1]:
import inspect
from qiskit_aer.primitives import EstimatorV2

print(inspect.signature(EstimatorV2))
est = EstimatorV2()
print("Has set_options?", hasattr(est, "set_options"))
print("Has options?", hasattr(est, "options"))
print("options repr:", est.options if hasattr(est, "options") else None)

(*, options: 'dict | None' = None)
Has set_options? False
Has options? True
options repr: Options(default_precision=0.0, backend_options={}, run_options={})


In [2]:
est = EstimatorV2()
est.options.shots = 256

In [3]:
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import n_local

# --- 1) Define a simple 2-qubit Hamiltonian H ---
# H = 1.0 * (Z ⊗ Z) + 0.5 * (X ⊗ I) + 0.5 * (I ⊗ X)
H = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

# --- 2) Define a small hardware-efficient ansatz (parameterized circuit family) ---
# Note 1: TwoLocal is deprecated in Qiskit 2.1+ but still works. We keep it for minimal first exposure.
# Note 2: .decompose() expands the template into primitive gate instructions, which is important for Aer 
# compatibility.
ansatz = n_local(
    num_qubits=2,
    rotation_blocks="ry",
    entanglement_blocks="cx",
    reps=1,
).decompose()

print("Ansatz gate counts:", ansatz.count_ops())


Ansatz gate counts: OrderedDict([('r', 4), ('cx', 1)])


## Baseline: exact expectation values (StatevectorEstimator)

This baseline removes sampling noise entirely by evaluating ⟨H⟩ exactly using a statevector-based estimator.
It provides a reference for what “converged” looks like in the noise-free setting.


In [4]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA, SPSA

optimizer_exact = COBYLA(maxiter=100)
estimator_exact = StatevectorEstimator()

vqe_exact = VQE(estimator=estimator_exact, ansatz=ansatz, optimizer=optimizer_exact)
result_exact = vqe_exact.compute_minimum_eigenvalue(H)

E_exact = float(np.real(result_exact.eigenvalue))
print("Exact (StatevectorEstimator) energy:", E_exact)


Exact (StatevectorEstimator) energy: -1.4142118043492977


## Finite-shot VQE: sampling noise and optimizer sensitivity

We now estimate expectation values using a finite number of shots (sampling).
We repeat the same experiment multiple times to measure run-to-run variability.

Then we compare two optimizers under the *same* Hamiltonian/ansatz/shots:
- COBYLA (deterministic, gradient-free)
- SPSA (stochastic, noise-tolerant)


In [5]:
from qiskit_aer.primitives import EstimatorV2

from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA, SPSA

# ----- Qiskit VQE class (Aer) -----

# Minimal Variational Quantum Eigensolver (VQE) on a 2-qubit Hamiltonian.
# Goal: run VQ end-to-end and print an estimated ground-state energy.
#
# This cell uses an Aer-based estimator. In the 08_first_vqe_end_to_end (same repo), 
# this cell defaults to an *exact* statevector-based estimator (deterministic, no shot noise),
# and optionally uses an Aer-based estimator if it is available AND compatible (it is, in our case).


shots = 256
num_runs = 3
maxiter = 100

def run_optimizer_experiment(optimizer, optimizer_name: str) -> list[float]:
    """Run VQE multiple times with the same settings and return final energies."""
    energies = []
    print(f"\n--- Optimizer: {optimizer_name} | shots={shots} | maxiter={maxiter} ---")
    for run in range(num_runs):
        estimator = EstimatorV2()
        # IMPORTANT: in this Aer version, shots are set via estimator.options
        estimator.options.shots = shots

        vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)
        result = vqe.compute_minimum_eigenvalue(H)
        energy = float(np.real(result.eigenvalue))
        energies.append(energy)
        print(f"Run {run+1}: energy = {energy}")

    print("Energies:", energies)
    print("Mean:", float(np.mean(energies)))
    print("Std :", float(np.std(energies)))
    return energies

energies_cobyla = run_optimizer_experiment(COBYLA(maxiter=maxiter), "COBYLA")
energies_spsa   = run_optimizer_experiment(SPSA(maxiter=maxiter), "SPSA")

print("\n=== Summary (lower is better) ===")
print(f"Exact baseline (StatevectorEstimator): {E_exact}")
print(f"COBYLA mean ± std: {float(np.mean(energies_cobyla))} ± {float(np.std(energies_cobyla))}")
print(f"SPSA   mean ± std: {float(np.mean(energies_spsa))} ± {float(np.std(energies_spsa))}")


--- Optimizer: COBYLA | shots=256 | maxiter=100 ---
Run 1: energy = -1.4141645878812592
Run 2: energy = -1.4142135370567064


Run 3: energy = -1.4142134875018177
Energies: [-1.4141645878812592, -1.4142135370567064, -1.4142134875018177]
Mean: -1.4141972041465944
Std : 2.3063191268567414e-05

--- Optimizer: SPSA | shots=256 | maxiter=100 ---
Run 1: energy = -1.4139375279334718
Run 2: energy = -1.2348355966197975
Run 3: energy = -1.392437672282505
Energies: [-1.4139375279334718, -1.2348355966197975, -1.392437672282505]
Mean: -1.3470702656119247
Std : 0.0798457957193422

=== Summary (lower is better) ===
Exact baseline (StatevectorEstimator): -1.4142118043492977
COBYLA mean ± std: -1.4141972041465944 ± 2.3063191268567414e-05
SPSA   mean ± std: -1.3470702656119247 ± 0.0798457957193422


### Interpretation: VQE behavior under exact vs finite-shot evaluation

- **Exact evaluation (statevector):**  
  With exact expectation values, VQE behaves deterministically. For this toy Hamiltonian,
  a shallow ansatz (reps=1) already reaches the variational optimum, and optimizer choice
  (e.g., COBYLA vs SPSA) has little impact on the final energy.

- **Finite-shot evaluation:**  
  When expectation values are estimated from a finite number of shots, the VQE objective
  becomes stochastic. Repeated runs with identical Hamiltonian, ansatz, and optimizer
  settings converge to slightly different energies, reflecting sampling noise in ⟨H⟩.

- **SPSA vs COBYLA under noise:**  
  SPSA is explicitly designed to tolerate noisy objective evaluations via stochastic
  gradient estimates. In contrast, deterministic optimizers such as COBYLA can become more
  sensitive—or brittle—when energy estimates fluctuate due to sampling noise.

- **Ansatz depth in the presence of noise:**  
  Increasing ansatz depth can improve expressivity in principle, but on noisy or
  shot-limited backends it may degrade performance by increasing circuit depth,
  measurement variance, and effective noise in the optimization landscape.

Overall, these results illustrate that VQE performance in the NISQ regime is governed not
only by the Hamiltonian and ansatz, but by the *interaction* between circuit depth,
measurement statistics, noise, and optimizer choice.